# VayuNetra E2 — Dense-Coverage Models (Colab/Kaggle GPU)

**Owner: Sejal.** Trains the two E2 models that turn ~40 ground stations into a full-city 1 km PM2.5 field:

1. **AOD→PM2.5 regressor** (LightGBM) — satellite AOD + meteorology → surface PM2.5.
2. **1 km downscaling CNN** (PyTorch) — learned super-resolution using a land-use covariate.

The module code (`ml/coverage/`) is identical to what runs CPU-only in the API/tests — this notebook just runs it on GPU and (when wired) swaps the synthetic generators for **real Earth-Engine AOD × CPCB stations**. Reports the honest Validation #7 numbers: AOD→PM2.5 RMSE and downscaling **skill vs plain interpolation** on held-out data.

In [ ]:
# 1. Environment
!git clone https://github.com/omkarrr88/VayuNetra.git || true
%cd VayuNetra
!pip -q install torch lightgbm scikit-learn h3 numpy pyyaml
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 2. AOD→PM2.5 regressor — held-out validation
from ml.coverage import aod_pm25
model, metrics = aod_pm25.train_and_validate()
print('AOD→PM2.5 held-out:', metrics.as_dict())
#  -> real data: replace aod_pm25.synth_pairs() with EE AOD + met × CPCB PM2.5 station pairs

In [ ]:
# 3. 1 km downscaling CNN — skill vs bilinear (GPU accelerates training)
from ml.coverage import downscale
X, y = downscale.make_dataset(n=1024, fine=48)   # larger set for GPU
if torch.cuda.is_available():
    X, y = X.cuda(), y.cuda()
cut = int(0.8*len(X))
cnn = downscale.train(X[:cut], y[:cut], epochs=120)
print('Downscale held-out:', downscale.evaluate(cnn, X[cut:], y[cut:]))
#  skill_vs_bilinear > 0  =>  the CNN adds genuine sub-grid information

In [ ]:
# 4. Build + save a city dense field (feeds /coverage + the map toggle)
from ml.coverage import build_dense_field
field = build_dense_field('delhi', bbox=(76.84, 28.40, 77.35, 28.88), base_pm25=110.0)
print('cells:', field['n_cells'], '| stats:', field['stats'], '| validation:', field['validation'])
# torch.save(cnn.state_dict(), 'ml/coverage/artifacts/downscale_cnn.pt')  # checkpoint to Storage/R2

## Wiring real data (Stage-2 integration)

- **AOD:** Earth Engine `MODIS/061/MCD19A2_GRANULES` (AOD 1 km) or Sentinel-5P, sampled at station H3 cells → replace `aod_pm25.synth_pairs`.
- **Stations:** CPCB CAAQMS PM2.5 (the `measurements` table) as the regression target + downscaler anchors.
- **Land-use covariate:** OSM road/built-up density (already ingested by `connectors/osm_sources.py`) rasterised to the fine grid.
- **Validation:** hold out whole stations (spatial CV) → report RMSE at unseen locations; that is the number for the deck, never a fit on training cells.